division des bassins de vie en fonction de leur densité d'habitants

La densité de population (hab/km²) est le meilleur indicateur du prix au m². La corrélation entre densité locale et prix DVF est nettement plus forte qu'avec le volume total d'habitants.

Le seuil de densité recommandé : 1 500 \hab/km2 : Zone urbaine dense (prix tirés vers le haut par l'habitat collectif, forte tension foncière).

Pour Paris (75), la densité dépasse 10 000 \hab/km^2 dans TOUS les arrondissements. La densité seule ne suffira donc pas à sous-découper Paris (elle les classera tous en "très dense"). Pour Paris, Lyon et Marseille, le découpage par arrondissement direct (ou par regroupement d'arrondissements adjacents) est déjà la sous-unité idéale pour DVF.

**Pourquoi filter sur la population et pas sur la densité de population**

Filtrer sur la population totale du BV au lieu de sa densité moyenne s'explique par deux raisons majeures de géographie urbaine :

1. L'effet de dilution administrative (le piège de la moyenne) :
La densité moyenne d'un bassin de vie dépend des frontières administratives des communes rurales qui ont été rattachées autour de la ville-centre.

- Exemple concret : Le BV de Toulouse rassemble 116 communes sur $1\ 355\text{ km}^2$. Sa densité moyenne n'est que de 838 hab/km² (ce qui semble faible), mais le BV compte 1,13 million d'habitants et contient la ville hyper-dense de Toulouse.

- Si tu avais appliqué un filtre sur la densité moyenne du BV ($\ge 1\ 500\text{ hab/km}^2$), tu aurais exclu des métropoles majeures comme Toulouse, Bordeaux ($814\text{ hab/km}^2$) ou Nantes ($1\ 049\text{ hab/km}^2$).

2. La masse critique du marché immobilier :
Un volume élevé d'habitants (ex. $> 100\ 000\text{ hab.}$) garantit un réseau urbain structuré avec de réels sous-marchés (un cœur hyper-dense vs une banlieue pavillonnaire vs du rural éloigné).

**En résumé :** On utilise la population totale pour cibler les grandes agglomérations, puis la densité communale/infra-communale pour découper l'intérieur de ces agglomérations.

In [101]:
import pandas as pd

# 1. Chargement de la table des communes
df_communes = pd.read_csv(r"C:\Users\gabi9\Documents\communes-france-2026.csv", dtype={'code_insee': str, 'code_bassin_de_vie': str})

C:\Users\gabi9\AppData\Local\Temp\ipykernel_8504\17847354.py:4: DtypeWarning: Columns (11: dep_code, 13: canton_code, 15: epci_code, 23: code_insee_centre_zone_emploi, 24: code_unite_urbaine, 33: code_insee_centre_aire_attraction) have mixed types. Specify dtype option on import or set low_memory=False.
  df_communes = pd.read_csv(r"C:\Users\gabi9\Documents\communes-france-2026.csv", dtype={'code_insee': str, 'code_bassin_de_vie': str})


In [103]:
# 2. Calcul de la population totale par bassin de vie
pop_bv = df_communes.groupby('code_bassin_de_vie')['population'].transform('sum')
df_communes['pop_bv_totale'] = pop_bv

# 3. Fonction d'attribution des sous-bassins
def attribuer_sous_bassin(row):
    # Si le BV fait moins de 100 000 habitants, on conserve le BV d'origine
    if row['pop_bv_totale'] < 100000:
        return row['code_bassin_de_vie']
    
    # Pour les grands BV (>= 100k hab), on découpe selon la grille de densité
    # Grille 1 & 2 : Grands centres et centres intermédiaires
    if row['grille_densite'] in [1, 2]:
        suffixe = "1"
    # Grille 3 & 4 : Ceintures urbaines et petites villes
    elif row['grille_densite'] in [3, 4]:
        suffixe = "2"
    # Grille 5, 6, 7 : Rural
    else:
        suffixe = "3"
        
    return f"{row['code_bassin_de_vie']}_{suffixe}"

df_communes['sous_bassin_de_vie'] = df_communes.apply(attribuer_sous_bassin, axis=1)

In [104]:
# 4. Table des populations légales des 45 arrondissements
pop_arrondissements = {
    # Paris (75101 à 75120)
    '75101': 15919, '75102': 21130, '75103': 34120, '75104': 28088,
    '75105': 57380, '75106': 39625, '75107': 47970, '75108': 35190,
    '75109': 59250, '75110': 83450, '75111': 142500, '75112': 140100,
    '75113': 177800, '75114': 133400, '75115': 227700, '75116': 162200,
    '75117': 163900, '75118': 188500, '75119': 181600, '75120': 189800,
    
    # Lyon (69381 à 69389)
    '69381': 29500, '69382': 30900, '69383': 101700, '69384': 35700,
    '69385': 48800, '69386': 52100, '69387': 82600, '69388': 86300, '69389': 51500,
    
    # Marseille (13201 à 13216)
    '13201': 39500, '13202': 23600, '13203': 45100, '13204': 49100,
    '13205': 45800, '13206': 42700, '13207': 34800, '13208': 82100,
    '13209': 76400, '13210': 56500, '13211': 57300, '13212': 61200,
    '13213': 92300, '13214': 61800, '13215': 79400, '13216': 38400
}

# Mapping Arrondissement -> Commune Parente
parent_map = {}
for code in pop_arrondissements:
    if code.startswith('751'):
        parent_map[code] = '75056'
    elif code.startswith('6938'):
        parent_map[code] = '69123'
    elif code.startswith('132'):
        parent_map[code] = '13055'

# 5. Création des nouvelles lignes d'arrondissements
lignes_arrondissements = []
for code_arm, pop_arm in pop_arrondissements.items():
    code_parent = parent_map[code_arm]
    
    # Extraction sécurisée du parent
    parent_matches = df_communes[df_communes['code_insee'] == code_parent]
    if not parent_matches.empty:
        parent_row = parent_matches.iloc[0]
        row_arm = parent_row.copy()
        row_arm['code_insee'] = code_arm
        
        # Vérification de l'existence de la colonne nom_standard
        if 'nom_standard' in row_arm:
            row_arm['nom_standard'] = f"{parent_row['nom_standard']} Arrondissement {code_arm[-2:]}"
            
        row_arm['population'] = pop_arm
        row_arm['sous_bassin_de_vie'] = code_arm
        lignes_arrondissements.append(row_arm)

# 6. Assemblage du référentiel final
df_referentiel = pd.concat([df_communes, pd.DataFrame(lignes_arrondissements)], ignore_index=True)

In [105]:
print(df_referentiel.columns)

Index(['code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de',
       'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte',
       'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'canton_code',
       'canton_nom', 'epci_code', 'epci_nom', 'code_postal', 'codes_postaux',
       'academie_code', 'academie_nom', 'zone_emploi', 'zone_emploi_nom',
       'code_insee_centre_zone_emploi', 'code_unite_urbaine',
       'nom_unite_urbaine', 'taille_unite_urbaine',
       'type_commune_unite_urbaine', 'statut_commune_unite_urbaine',
       'code_aire_attraction', 'nom_aire_attraction',
       'categorie_aire_attraction', 'taille_aire_attraction',
       'code_insee_centre_aire_attraction', 'code_bassin_de_vie',
       'nom_bassin_de_vie', 'type_bassin_de_vie', 'population',
       'superficie_hectare', 'superficie_km2', 'densite',
       'superficie_cartographique_ign_hectare',
       'superficie_cartographique_ign_km2', 'densite_cartographique',
       'superficie_c

In [106]:
df_referentiel=df_referentiel.drop(columns=['nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de',
       'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte',
       'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'canton_code',
       'canton_nom', 'epci_code', 'epci_nom', 'code_postal', 'codes_postaux',
       'academie_code', 'academie_nom', 'zone_emploi', 'zone_emploi_nom',
       'code_insee_centre_zone_emploi', 'code_unite_urbaine',
       'nom_unite_urbaine', 'taille_unite_urbaine',
       'type_commune_unite_urbaine', 'statut_commune_unite_urbaine',
       'code_aire_attraction', 'nom_aire_attraction',
       'categorie_aire_attraction', 'taille_aire_attraction',
       'code_insee_centre_aire_attraction','superficie_hectare', 'superficie_km2','superficie_cartographique_ign_hectare',
       'superficie_cartographique_ign_km2', 'densite_cartographique',
       'superficie_cadastrale_ign_hectare', 'superficie_cadastrale_ign_km2',
       'densite_cadastrale', 'altitude_moyenne', 'altitude_minimale',
       'altitude_maximale', 'latitude_mairie', 'longitude_mairie',
       'latitude_centre', 'longitude_centre','niveau_equipements_services',
       'niveau_equipements_services_texte', 'qpv_nombre', 'gentile',
       'url_wikipedia', 'url_villedereve'])

In [107]:
print(df_referentiel.columns)

Index(['code_insee', 'code_bassin_de_vie', 'nom_bassin_de_vie',
       'type_bassin_de_vie', 'population', 'densite', 'grille_densite',
       'grille_densite_texte', 'pop_bv_totale', 'sous_bassin_de_vie'],
      dtype='str')


In [127]:
# 7. Suppression des 3 communes mères pour éliminer tout risque de doublon
codes_a_supprimer = ['75056', '69123', '13055']
df_referentiel = df_referentiel[~df_referentiel['code_insee'].isin(codes_a_supprimer)].copy()

In [128]:
df_referentiel.to_csv(r"C:\Users\gabi9\Documents\bv_scindes_densite_pop.csv", index=False, encoding='utf-8')

In [129]:
## jointure liaison commune bv2022 et le df composé des bassins de vie scindés en fonction de leur densité de pop

# 1. Chargement des fichiers en forçant le type texte (str) pour conserver les zéro initiaux
df_liaison = pd.read_csv(r'C:\Users\gabi9\Documents\liaison_commune_bv2022.csv', dtype=str)
df_bv = pd.read_csv(r'C:\Users\gabi9\Documents\bv_scindes_densite_pop.csv', dtype=str)

# 2. Formater les codes INSEE sur 5 caractères (ex: '1001' -> '01001')
df_liaison['CODGEO_INI_clean'] = df_liaison['CODGEO_INI'].str.zfill(5)
df_bv['code_insee_clean'] = df_bv['code_insee'].str.zfill(5)

# 3. Fusion des dataframes sur le fichier de liaison
df_merged = pd.merge(
    df_liaison,
    df_bv,
    left_on=['CODGEO_INI_clean', 'BV2022'],
    right_on=['code_insee_clean', 'sous_bassin_de_vie'],
    how='left'
)


In [131]:
print(df_merged.columns)

Index(['CODGEO_INI', 'CODGEO_2026', 'LIBGEO_2026', 'BV2022', 'LIBBV2022',
       'TYPE_BV2022', 'CODGEO_INI_clean', 'code_insee', 'code_bassin_de_vie',
       'nom_bassin_de_vie', 'type_bassin_de_vie', 'population', 'densite',
       'grille_densite', 'grille_densite_texte', 'pop_bv_totale',
       'sous_bassin_de_vie', 'code_insee_clean'],
      dtype='str')


In [132]:
# 4. Suppression des colonnes de nettoyage temporaires et autres colonnes indicatives
df_merged = df_merged.drop(columns=['LIBGEO_2026', 'BV2022', 'LIBBV2022',
       'TYPE_BV2022', 'CODGEO_INI_clean', 'code_insee', 'code_bassin_de_vie',
       'nom_bassin_de_vie', 'type_bassin_de_vie','code_insee_clean','population', 'densite', 'grille_densite',
       'grille_densite_texte', 'pop_bv_totale'])

In [133]:
print(df_merged.columns)

Index(['CODGEO_INI', 'CODGEO_2026', 'sous_bassin_de_vie'], dtype='str')


In [134]:
# 5. renommer les colonnes
df_merged = df_merged.rename(columns={'sous_bassin_de_vie': 'BV2022_scinde'})

In [135]:
df_merged.head()

,CODGEO_INI,CODGEO_2026,BV2022_scinde
0,01001,01001,01093
1,01002,01002,01004
2,01004,01004,01004
3,01005,01005,01333
4,01006,01006,01034


In [137]:
# 6. Export du fichier fusionné
df_merged.to_csv('liaison_bv_scinde.csv', index=False, encoding='utf-8-sig')